# ASH-KV vs Baseline SGLang Benchmark

This notebook measures the concurrency limits and latency impact of the ASH-KV INT8 HiCache interception versus the baseline SGLang CPU offload.

## Phase 1: Baseline Test (No ASH-KV)

**Step 1:** In your server terminal, launch standard SGLang WITHOUT the ASH-KV script:
```bash
python -m sglang.launch_server --model-path Qwen/Qwen2.5-7B-Instruct --port 30000 --mem-fraction-static 0.3 --hicache-write-policy write_through
```
Wait for the server to start, then run the cells below.

In [ ]:
import requests
import time
import concurrent.futures

ENDPOINT = "http://localhost:30000/v1/chat/completions"
MODEL = "Qwen/Qwen2.5-7B-Instruct"

def make_prompt(i, role="user", mult=200):
    return {
        "model": MODEL,
        "messages": [
            {"role": role, "content": f"Repeat this exactly {i}: " + "The quick brown fox jumps over the lazy dog. " * mult}
        ],
        "max_tokens": 50,
        "temperature": 0.0,
    }

def measure_request(i):
    start = time.time()
    try:
        r = requests.post(ENDPOINT, json=make_prompt(i), timeout=60)
        latency = time.time() - start
        if r.status_code == 200:
            return {'status': 'success', 'latency': latency}
        else:
            return {'status': f'failed_{r.status_code}', 'latency': latency}
    except Exception as e:
        return {'status': 'error', 'latency': time.time() - start}

# Run concurrency test
print("Running baseline concurrency test...")
CONCURRENCY = 15
start_total = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=CONCURRENCY) as pool:
    baseline_results = list(pool.map(measure_request, range(CONCURRENCY)))
total_time_baseline = time.time() - start_total

baseline_success = sum(1 for r in baseline_results if r['status'] == 'success')
baseline_avg_latency = sum(r['latency'] for r in baseline_results if r['status'] == 'success') / max(1, baseline_success)
print(f"Baseline: {baseline_success}/{CONCURRENCY} successful. Avg Latency: {baseline_avg_latency:.2f}s. Total time: {total_time_baseline:.2f}s")

## Phase 2: ASH-KV Test

**Step 2:** Stop the baseline server (`Ctrl+C`). Now, launch the ASH-KV server using our wrapper:
```bash
PYTHONPATH=. python tests/run_real_sglang_server.py --model Qwen/Qwen2.5-7B-Instruct --port 30000
```
Wait for the server to say `--- ENGINE STARTED ---`, then run the cells below.

In [ ]:
# Run concurrency test for ASH-KV
print("Running ASH-KV concurrency test...")
start_total = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=CONCURRENCY) as pool:
    ashkv_results = list(pool.map(measure_request, range(CONCURRENCY)))
total_time_ashkv = time.time() - start_total

ashkv_success = sum(1 for r in ashkv_results if r['status'] == 'success')
ashkv_avg_latency = sum(r['latency'] for r in ashkv_results if r['status'] == 'success') / max(1, ashkv_success)
print(f"ASH-KV: {ashkv_success}/{CONCURRENCY} successful. Avg Latency: {ashkv_avg_latency:.2f}s. Total time: {total_time_ashkv:.2f}s")

## Phase 3: Comparison

In [ ]:
print("=== BENCHMARK RESULTS ===")
print(f"Concurrency Level: {CONCURRENCY}")
print("-" * 30)
print("BASELINE (BF16 CPU Offload):")
print(f"  Success Rate: {baseline_success}/{CONCURRENCY}")
print(f"  Avg Latency:  {baseline_avg_latency:.2f}s")
print(f"  Total Time:   {total_time_baseline:.2f}s")
print("-" * 30)
print("ASH-KV (INT8 Compression):")
print(f"  Success Rate: {ashkv_success}/{CONCURRENCY}")
print(f"  Avg Latency:  {ashkv_avg_latency:.2f}s")
print(f"  Total Time:   {total_time_ashkv:.2f}s")
print("-" * 30)
if ashkv_success > baseline_success:
    print(f"🏆 ASH-KV survived {ashkv_success - baseline_success} more concurrent requests without OOM/Failure!")
elif total_time_ashkv < total_time_baseline and ashkv_success == baseline_success:
    print(f"🏆 ASH-KV was {total_time_baseline - total_time_ashkv:.2f}s faster at the same concurrency level!")